# VMTF-Net — Chạy trên Google Colab Extension (VS Code / Antigravity)

**Cách chạy:**
1. Ở góc trên bên phải của file notebook này, bấm **Select Kernel** → chọn **Colab** (GPU T4 / V100 / A100) hoặc **Python local**.
2. Nếu trong `data/` đã có `vmtf_candidates.csv` (đã điền cột `label` `0`/`1`) và `images/`, Cell 3 sẽ tự chạy `validate_and_preprocess_dataset.py` để tạo `train.csv`, `val.csv`, `test.csv`.
3. Nếu chưa có dữ liệu thật, Cell 3 tự tạo 4 mẫu giả lập để test pipeline.

In [ ]:
# 1. Kiểm tra GPU trên Runtime
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "Đang chạy CPU runtime (hãy đổi Kernel sang GPU nếu muốn train nhanh)"

In [ ]:
# 2. Thiết lập thư mục làm việc (Colab VM hoặc Local) và cài thư viện bổ sung
import os
if os.path.exists('/content'):
    %cd /content
    if not os.path.exists('/content/vietnamese-multimodal-fake-news-detection'):
        !git clone -q https://github.com/kascik04/vietnamese-multimodal-fake-news-detection.git
    else:
        !git -C /content/vietnamese-multimodal-fake-news-detection pull -q
    %cd /content/vietnamese-multimodal-fake-news-detection
    !pip install -q pyvi imagehash
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print('Working dir:', os.getcwd())
!git log --oneline -1

In [ ]:
# 3. Tự động xử lý data/vmtf_candidates.csv -> train/val/test (hoặc tạo 4 mẫu test nếu chưa có dữ liệu)
import os, sys, subprocess
import pandas as pd
from PIL import Image, ImageDraw

os.makedirs('data/images', exist_ok=True)
IS_SMOKE_TEST = False

if not os.path.exists('data/train.csv') and os.path.exists('data/vmtf_candidates.csv'):
    df_raw = pd.read_csv('data/vmtf_candidates.csv')
    labeled = df_raw[df_raw['label'].isin([0, 1, '0', '1'])]
    if len(labeled) >= 6 and labeled['label'].nunique() >= 2:
        print(f'✅ Tìm thấy {len(labeled)}/{len(df_raw)} mẫu đã gán nhãn trong data/vmtf_candidates.csv -> đang chia train/val/test...')
        subprocess.run([
            sys.executable, 'scripts/validate_and_preprocess_dataset.py',
            '--input_csv', 'data/vmtf_candidates.csv',
            '--image_dir', 'data/images',
            '--output_dir', 'data',
        ], check=True)
    else:
        raise ValueError(
            f'data/vmtf_candidates.csv mới có {len(labeled)}/{len(df_raw)} dòng được gán nhãn (cần đủ cả nhãn 0 và 1). '
            'Hãy điền cột label (0 = tin thật, 1 = tin giả) trước khi chạy!'
        )

if not os.path.exists('data/train.csv'):
    IS_SMOKE_TEST = True
    print('⚠️ Chưa thấy data/train.csv hay data/vmtf_candidates.csv -> tạo 4 mẫu giả lập để smoke-test pipeline.')
    with open('data/train.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_001,Bộ Y Tế chính thức bác bỏ tin đồn thất thiệt lan truyền trên mạng xã hội sáng nay.,news_1.jpg,1250,340,120,15,30,0\n'
            'post_002,Cảnh báo khẩn cấp: Phương pháp chữa bách bệnh bằng lá cây chưa qua kiểm chứng khoa học!,news_2.jpg,89000,15400,45000,31150,35600,1\n'
            'post_003,Chính phủ phê duyệt dự án phát triển công nghệ cao và trí tuệ nhân tạo quốc gia.,news_3.jpg,4500,820,910,10,45,0\n'
            'post_004,Sốc: Hình ảnh sinh vật lạ xuất hiện tại vùng biển miền Trung gây hoang mang dư luận.,news_4.jpg,52000,8700,19500,13000,28600,1\n'
        )
    with open('data/val.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_001,Bộ Y Tế chính thức bác bỏ tin đồn thất thiệt lan truyền trên mạng xã hội sáng nay.,news_1.jpg,1250,340,120,15,30,0\n'
            'post_002,Cảnh báo khẩn cấp: Phương pháp chữa bách bệnh bằng lá cây chưa qua kiểm chứng khoa học!,news_2.jpg,89000,15400,45000,31150,35600,1\n'
        )
    with open('data/test.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_003,Chính phủ phê duyệt dự án phát triển công nghệ cao và trí tuệ nhân tạo quốc gia.,news_3.jpg,4500,820,910,10,45,0\n'
            'post_004,Sốc: Hình ảnh sinh vật lạ xuất hiện tại vùng biển miền Trung gây hoang mang dư luận.,news_4.jpg,52000,8700,19500,13000,28600,1\n'
        )
    colors = {'news_1.jpg': (30, 136, 229), 'news_2.jpg': (229, 57, 53), 'news_3.jpg': (67, 160, 71), 'news_4.jpg': (251, 140, 0)}
    for name, rgb in colors.items():
        p = os.path.join('data/images', name)
        if not os.path.exists(p):
            img = Image.new('RGB', (224, 224), rgb)
            ImageDraw.Draw(img).rectangle([40, 40, 184, 184], outline=(255, 255, 255), width=4)
            img.save(p)

for split in ('train', 'val', 'test'):
    p = f'data/{split}.csv'
    if os.path.exists(p):
        df_s = pd.read_csv(p)
        print(f'{split}.csv: {len(df_s)} mẫu | phân bố nhãn: {df_s["label"].value_counts().to_dict()}')

In [ ]:
# 4. Chạy Offline Unit Tests (tùy chọn) & Train VMTF-Net
import os, pandas as pd

RUN_UNIT_TESTS = False
if RUN_UNIT_TESTS and os.path.exists('test_step1.py'):
    !python test_step1.py && python test_step2.py && python test_step3.py && python test_step4.py && python test_step5.py

n_train = len(pd.read_csv('data/train.csv'))
EXP_DIR = 'checkpoints/colab_ext_smoke_test' if IS_SMOKE_TEST else 'checkpoints/colab_ext_proposed_seed42'
BATCH   = 2 if n_train < 16 else 16
EPOCHS  = 2 if IS_SMOKE_TEST else 10
N_BOOT  = 50 if IS_SMOKE_TEST else 1000
OVERWRITE = False  # True nếu muốn train lại từ đầu, xóa checkpoint cũ
CACHE_DIR = '/content/cache' if os.path.exists('/content') else 'data/cache'

last_ckpt = f'{EXP_DIR}/last_checkpoint.pt'
resume_flag = '--overwrite' if OVERWRITE else (f'--resume {last_ckpt}' if os.path.exists(last_ckpt) else '')

!python -m vmtf_net.train \
    --train_csv data/train.csv \
    --val_csv data/val.csv \
    --test_csv data/test.csv \
    --image_dir data/images \
    --cache_dir {CACHE_DIR} \
    --fusion_type hybrid_proposed \
    --consistency_lambda 0.1 \
    --social_preset basic \
    --batch_size {BATCH} \
    --epochs {EPOCHS} \
    --lr 2e-5 \
    --head_lr 1e-4 \
    --n_bootstrap {N_BOOT} \
    --checkpoint_dir {EXP_DIR} \
    --no_tqdm {resume_flag}

In [ ]:
# 5. Xem kết quả train & thông số GPU
import json
import pandas as pd

display(pd.read_csv(f'{EXP_DIR}/history.csv'))
print('--- run_info.json ---')
print(json.dumps(json.load(open(f'{EXP_DIR}/run_info.json')), indent=2, ensure_ascii=False))
print('--- test metrics ---')
print(json.dumps(json.load(open(f'{EXP_DIR}/results.json'))['test']['metrics'], indent=2, ensure_ascii=False))